In [ ]:
import os, glob, gc
import numpy as np
import xarray as xr
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor, as_completed

# ✅ 导入计算函数
try:
    from aostools_functions import ComputeEPfluxDiv
except ImportError:
    raise ImportError("请确保 'aostools_functions.py' 与此脚本在同一目录下！")

# ---------------- 配置路径 (测试版) ----------------
INPUT_DIR   = "/home/weiji/restart_exam/longrun_B2000WCN_withchem_data/EP_Daily_Inputs"
# 自动在原目录名后加 
OUT_DIR     = "/home/weiji/restart_exam/code/20260122StructureAnalysis/result"
OUT_FILE_NC = os.path.join(OUT_DIR, "EPFLUX_206yr_Daily_Series_Full.nc")

# ---------------- 设置 ----------------
PLEV_STD_PA = np.array([
    10, 50, 100, 200, 300, 500, 1000, 2000, 3000, 5000, 7000, 10000, 15000,
    20000, 25000, 30000, 40000, 50000, 60000, 70000, 85000, 92500, 100000
], dtype=float)

LAT_RANGE = (40.0, 80.0) 
DO_UBAR    = False   
WAVE       = -1      
MAX_WORKERS = 32  # 🚀 你有128核，如果内存够大(如>256GB)，可以改为 64 或 128

# ---------------- 辅助函数 ----------------
def compute_pressure_mid(ds: xr.Dataset) -> xr.DataArray:
    return ds["hyam"] * ds["P0"] + ds["hybm"] * ds["PS"]

def interp_profile_logp_4d(v_hyb, p_hyb, p_tgt_pa):
    p_tgt_pa = np.asarray(p_tgt_pa, float)
    def _interp_col(vcol, pcol):
        m = np.isfinite(vcol) & np.isfinite(pcol) & (pcol > 0)
        if m.sum() < 2: return np.full(p_tgt_pa.shape, np.nan, float)
        p_use, v_use = pcol[m], vcol[m]
        idx = np.argsort(p_use)
        return np.interp(np.log(p_tgt_pa), np.log(p_use[idx]), v_use[idx], left=np.nan, right=np.nan)

    dask_gufunc_kwargs = {"output_sizes": {"plev": len(p_tgt_pa)}}
    out = xr.apply_ufunc(
        _interp_col, v_hyb, p_hyb,
        input_core_dims=[["lev"], ["lev"]],
        output_core_dims=[["plev"]],
        vectorize=True, dask="parallelized", output_dtypes=[float],
        dask_gufunc_kwargs=dask_gufunc_kwargs
    )
    return out.assign_coords(plev=("plev", p_tgt_pa))

def coslat_weighted_mean(da, lat_name="lat"):
    lat = da[lat_name]
    w = np.cos(np.deg2rad(lat))
    return da.weighted(w).mean(lat_name, skipna=True)

# ---------------- 单文件处理函数 (用于并行) ----------------
def process_one_file(fpath):
    """处理单个年份的文件并返回计算结果"""
    try:
        with xr.open_dataset(fpath) as ds:
            if ds.sizes["time"] < 2: return None

            # 1. 垂直插值
            p_mid = compute_pressure_mid(ds)
            u_std = interp_profile_logp_4d(ds["U"], p_mid, PLEV_STD_PA)
            v_std = interp_profile_logp_4d(ds["V"], p_mid, PLEV_STD_PA)
            t_std = interp_profile_logp_4d(ds["T"], p_mid, PLEV_STD_PA)
            w_std = interp_profile_logp_4d(ds["OMEGA"]/100.0, p_mid, PLEV_STD_PA)

            # 2. 准备 Numpy 数组
            u_np = u_std.transpose("time", "plev", "lat", "lon").values
            v_np = v_std.transpose("time", "plev", "lat", "lon").values
            t_np = t_std.transpose("time", "plev", "lat", "lon").values
            w_np = w_std.transpose("time", "plev", "lat", "lon").values
            lat_np = ds["lat"].values

            # 3. 计算 EP Flux (ep2 是 Fz)
            _, ep2, _, _ = ComputeEPfluxDiv(
                lat=lat_np, pres=(PLEV_STD_PA/100.0),
                u=u_np, v=v_np, t=t_np, w=w_np,
                do_ubar=DO_UBAR, wave=WAVE
            )
            
            # 4. 封装与区域平均
            da_ep2 = xr.DataArray(
                ep2, 
                coords={"time": ds["time"], "plev": PLEV_STD_PA, "lat": lat_np}, 
                dims=("time", "plev", "lat")
            )
            
            lat0, lat1 = LAT_RANGE
            sel_lat = slice(lat1, lat0) if lat_np[0] > lat_np[-1] else slice(lat0, lat1)
            da_sub = da_ep2.sel(lat=sel_lat)
            
            # 返回计算好的 Profile (已经变成 1D time x 1D plev)
            result = coslat_weighted_mean(da_sub).compute()
            return result

    except Exception as e:
        return f"Error in {os.path.basename(fpath)}: {str(e)}"

# ---------------- 主程序 ----------------
def main():
    os.makedirs(OUT_DIR, exist_ok=True)
    
    files = sorted(glob.glob(os.path.join(INPUT_DIR, "*.nc")))
    if not files:
        print(f"❌ Error: No files found in {INPUT_DIR}")
        return

    print(f"🚀 Parallel Start: Using {MAX_WORKERS} workers for {len(files)} years...")
    
    pool_results = []
    
    # 使用进程池并行执行
    with ProcessPoolExecutor(max_workers=MAX_WORKERS) as executor:
        # 提交所有任务
        future_to_file = {executor.submit(process_one_file, f): f for f in files}
        
        # 使用 tqdm 显示进度
        for future in tqdm(as_completed(future_to_file), total=len(files), desc="Parallel Computing"):
            res = future.result()
            if isinstance(res, xr.DataArray):
                pool_results.append(res)
            elif res is not None:
                print(f"⚠️ {res}")

    # 汇总并保存
    if not pool_results:
        print("❌ Error: No data collected.")
        return

    print("📊 Finalizing: Concatenating and saving...")
    ds_final = xr.concat(pool_results, dim="time").sortby("time")
    
    ds_output = xr.Dataset({"Fz": ds_final})
    ds_output.attrs = {
        "description": "EP Flux (Fz) FULL Daily Time Series (Test)",
        "lat_range": f"{LAT_RANGE[0]}N-{LAT_RANGE[1]}N",
        "units": "m^2/s^2",
        "history": "Parallel processed test version"
    }

    ds_output.to_netcdf(OUT_FILE_NC)
    print(f"✅ Success! Test File saved to: {OUT_FILE_NC}")

if __name__ == "__main__":
    main()

In [ ]:
import os
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

# ============================================================
# Fig(f)-style plot: Fz standardized anomalies (blue-white-red, -4..4)
# Reference: 206-year Daily Full Series (Parallel Processed)
# Winter: 0007/11/01 - 0008/04/30 (model calendar)
# ============================================================

# ---------------- paths ----------------
# 指向您并行计算生成的 test 结果目录
OUT_DIR = "/home/weiji/restart_exam/code/20260122StructureAnalysis/result"
RESULT_DIR   = "/home/weiji/restart_exam/code/20260122StructureAnalysis/result"

# 1. 信号文件 (23年 run)
EP_NC   = f"{RESULT_DIR}/EPFLUX_BWCN002_stdplev_daily_lat_ALL.nc"
EHF_NC  = f"{RESULT_DIR}/EHF_BWCN002_vTprime_ALL_LAT_stdplev_time_lat.nc"

# 2. 气候态参考文件 (使用并行生成的 206年全量数据)
CLIM_NC = f"{OUT_DIR}/EPFLUX_206yr_Daily_Series_Full.nc"

# ✅ 输出图片路径
OUT_FIG = os.path.join(OUT_DIR, "Fig_f_Fz_std_anom_206yr_FullRef.png")

# ---------------- settings ----------------
t0, t1 = "0007-11-01", "0008-04-30"  # 目标窗口（包含4月）
LAT0, LAT1 = 40.0, 80.0
ZLIM = 4.0

# ---------------- helpers ----------------
def sel_latband(da, lat0=40., lat1=80., lat_name="lat"):
    lat = da[lat_name]
    dec = float(lat.values[0]) > float(lat.values[-1])
    slc = slice(lat1, lat0) if dec else slice(lat0, lat1)
    out = da.sel({lat_name: slc})
    if out.sizes.get(lat_name, 0) == 0:
        raise ValueError("Empty lat selection.")
    return out

def coslat_weighted_mean(da, lat_name="lat"):
    lat = da[lat_name]
    w = np.cos(np.deg2rad(lat))
    return da.weighted(w).mean(lat_name, skipna=True)

# ---------------- load & pre-process ----------------
print("📖 Loading datasets...")
ds_ep   = xr.open_dataset(EP_NC)
ds_ehf  = xr.open_dataset(EHF_NC)
ds_full = xr.open_dataset(CLIM_NC)

# 提取信号场并转换符号 (向上为正)
Fz_sig_up = -ds_ep["EP2_cart"] 
vT_sig    = ds_ehf["EHF_vTprime_stdplev"]

# 提取参考场 (并行代码输出已经是区域平均后的 Fz)
Fz_ref_up = -ds_full["Fz"]

# ---------------- signal selection ----------------
Fz_win = Fz_sig_up.sel(time=slice(t0, t1))
vT_win = vT_sig.sel(time=slice(t0, t1))

# 计算信号场的区域平均
Fz_40_80 = coslat_weighted_mean(sel_latband(Fz_win, LAT0, LAT1))
vT_40_80 = coslat_weighted_mean(sel_latband(vT_win, LAT0, LAT1))

nt = Fz_40_80.sizes["time"]
x = np.arange(nt)
time_str = np.array([str(tt)[:10] for tt in Fz_40_80["time"].values])

# ---------------- standardize using 206-yr Climatology ----------------
print("📊 Calculating standardized anomalies using 206-yr reference...")

# 1. 从 206 年全序列计算每日气候态 (1-366天)
clim_mu = Fz_ref_up.groupby("time.dayofyear").mean("time")
clim_sd = Fz_ref_up.groupby("time.dayofyear").std("time")

# 2. 匹配信号场每一天对应的气候态 (由于 ref 包含全年，这里不会报 KeyError)
sig_doy = Fz_40_80.time.dt.dayofyear
mu_matched = clim_mu.sel(dayofyear=sig_doy)
sd_matched = clim_sd.sel(dayofyear=sig_doy).where(lambda x: x > 1e-12)

# 3. 计算标准化距平
Fz_stdzd_vals = (Fz_40_80.values - mu_matched.values) / sd_matched.values
Fz_stdzd = xr.DataArray(Fz_stdzd_vals, coords=Fz_40_80.coords, dims=Fz_40_80.dims)

# ---------------- plotting arrays ----------------
plev = Fz_stdzd["plev"].values
Z_Fz = Fz_stdzd.transpose("plev", "time").values
Z_vT = vT_40_80.transpose("plev", "time").values

# ---------------- plot ----------------
print("🎨 Plotting...")
fig, ax = plt.subplots(figsize=(14, 4.8))

levels = np.linspace(-ZLIM, ZLIM, 17)
cmap = plt.get_cmap("RdBu_r")
norm = TwoSlopeNorm(vmin=-ZLIM, vcenter=0.0, vmax=ZLIM)

cf = ax.contourf(x, plev, Z_Fz, levels=levels, cmap=cmap, norm=norm, extend="both")

# 叠加 v'T' = 0 等值线 (虚线)
ax.contour(x, plev, Z_vT, levels=[0.0], colors="k", linestyles="--", linewidths=1.3)

# 坐标轴设置
ax.set_yscale("log")
ax.invert_yaxis()

# 设置 Y 轴刻度 (Pa 转 hPa)
yticks_pa = np.array([100000, 70000, 50000, 30000, 20000, 10000, 7000, 5000, 3000, 2000, 1000, 500, 300, 200, 100, 50, 10])
yticks_pa = yticks_pa[(yticks_pa >= float(plev.min())) & (yticks_pa <= float(plev.max()))]
ax.set_yticks(yticks_pa)
ax.set_yticklabels([f"{int(p/100)}" for p in yticks_pa])
ax.set_ylabel("Pressure (hPa)")

# 设置 X 轴时间刻度
tick_idx = np.linspace(0, nt-1, 8, dtype=int)
ax.set_xticks(tick_idx)
ax.set_xticklabels(time_str[tick_idx])
ax.set_xlabel("Model Date")

# 修改标题设置逻辑
title_str = "40–80°N $F_z$ Std Anomaly (Ref: 206-yr Full Daily Climatology)\n"
title_str += r"Winter 0007/0008 | Dashed: $\overline{v'T'}=0$"
ax.set_title(title_str)

cbar = fig.colorbar(cf, ax=ax, pad=0.02)
cbar.set_label("Standardized Anomaly ($\sigma$)")
cbar.set_ticks([-4, -2, 0, 2, 4])

plt.tight_layout()

# 保存
os.makedirs(os.path.dirname(OUT_FIG), exist_ok=True)
plt.savefig(OUT_FIG, dpi=200, bbox_inches="tight")
print(f"✅ Success! Figure saved to: {OUT_FIG}")
plt.show()

# 释放内存
ds_ep.close(); ds_ehf.close(); ds_full.close()